<a href="https://colab.research.google.com/github/iam4tart/speech-lab/blob/main/02-ctc-stt-from-scratch/train_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import random
import numpy as np

import torch
import torch.nn as nn

import torchaudio
import pandas as pd

from torch.utils.data import Dataset, DataLoader

In [ ]:
# setting device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
torch.backends.cudnn.benchmark = True

In [ ]:
# setting seed
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# config
CFG = {
    "sample_rate": 16000,
    "batch_size": 16,
    "epochs": 100,
    "lr": 5e-4,
    "embedding_dim": 32,
    "num_transformer_layers": 6,
    "num_heads": 4,
    "strides": (2, 2, 2, 2),
    "num_codebooks": 4,
    "codebook_size": 1024,
    "commitment_cost": 0.25,
    "vq_initial_weight": 10.0,
    "vq_final_weight": 0.5,
    "vq_warmup_steps": 1000,
}

In [ ]:
# download dataset
os.makedirs("./data", exist_ok=True)
!wget -q https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2
!tar -xjf LJSpeech-1.1.tar.bz2 -C ./data/
dataset_path = "./data/LJSpeech-1.1"
print(dataset_path)

./data/LJSpeech-1.1


In [ ]:
# read metadata
import os
import pandas as pd

path = "./data/LJSpeech-1.1"

metadata = pd.read_csv(
    os.path.join(path, "metadata.csv"),
    sep="|",
    header=None,
    names=["id", "text", "normalized_text"]
)

metadata = metadata.dropna(subset=["normalized_text"]).reset_index(drop=True)

metadata.head()

,id,text,normalized_text
0,LJ001-0001,"Printing, in the only sense with which we are ...","Printing, in the only sense with which we are ..."
1,LJ001-0002,in being comparatively modern.,in being comparatively modern.
2,LJ001-0003,For although the Chinese took impressions from...,For although the Chinese took impressions from...
3,LJ001-0004,"produced the block books, which were the immed...","produced the block books, which were the immed..."
4,LJ001-0005,the invention of movable metal letters in the ...,the invention of movable metal letters in the ...


In [ ]:
# load audio and text properly
wavs_path = os.path.join(path, "wavs")

audio_paths = [
    os.path.join(wavs_path, f"{fid}.wav") for fid in metadata["id"]
]

texts = [
    t.upper() for t in metadata["normalized_text"]
]

print(audio_paths[0], texts[0])

./data/LJSpeech-1.1/wavs/LJ001-0001.wav PRINTING, IN THE ONLY SENSE WITH WHICH WE ARE AT PRESENT CONCERNED, DIFFERS FROM MOST IF NOT FROM ALL THE ARTS AND CRAFTS REPRESENTED IN THE EXHIBITION


In [ ]:
import string

# vocab
characters = list(string.ascii_uppercase) + [" "]
blank_token = "<blank>"

vocab = characters + [blank_token]

# mappings
char_to_idx = {ch: idx for idx, ch in enumerate(vocab)}
idx_to_char = {idx: ch for ch, idx in char_to_idx.items()}

print("Vocab size:", len(vocab))

Vocab size: 28


In [ ]:
# tokenizer
def text_to_tokens(text):
  return [char_to_idx[c] for c in text if c in char_to_idx]

token_sequences = [text_to_tokens(t) for t in texts]

In [ ]:
# build dataset
class STTDataset(Dataset):
  def __init__(self, audio_paths, token_sequences):
    self.audio_paths = audio_paths
    self.token_sequences = token_sequences

  def __len__(self):
    return len(self.audio_paths)

  def __getitem__(self, idx):
    # load audio
    waveform, sr = torchaudio.load(self.audio_paths[idx])

    # mono
    if waveform.shape[0] > 1:
      waveform = waveform.mean(dim=0, keepdim=True) # (1, T)

    # resample to 16k
    if sr != 16000:
      waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)

    waveform = waveform.squeeze(0)  # (T,)

    # tokens
    tokens = torch.tensor(self.token_sequences[idx], dtype=torch.long)

    return waveform, tokens

In [ ]:
# collate function (needed for ctc_loss)
def collate_fn(batch):
  waveforms, tokens = zip(*batch)

  audio_lengths = torch.tensor([w.shape[0] for w in waveforms], dtype=torch.long)
  target_lengths = torch.tensor([len(t) for t in tokens], dtype=torch.long)

  waveforms = torch.nn.utils.rnn.pad_sequence(waveforms, batch_first=True)  # (B, T)
  tokens = torch.nn.utils.rnn.pad_sequence(tokens, batch_first=True)        # (B, U)

  return waveforms, tokens, audio_lengths, target_lengths

In [ ]:
# dataloader
dataset = STTDataset(
    audio_paths,
    token_sequences
)

loader = DataLoader(
    dataset,
    batch_size=CFG["batch_size"],
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True,
)

In [ ]:
class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, commitment_cost=0.25):
      super().__init__()

      self.embedding = nn.Embedding(num_embeddings, embedding_dim)
      nn.init.uniform_(self.embedding.weight, -0.1, 0.1)
      self.commitment_cost = commitment_cost

    def forward(self, x):
      B, T, D = x.shape
      flat_x = x.reshape(B*T, D)
      distances = torch.cdist(flat_x, self.embedding.weight, p=2)
      encoding_indices = torch.argmin(distances, dim=1)
      quantized = self.embedding(encoding_indices).view(B, T, D)

      # gradient flows into `quantized`
      # moves codebook vectors towards x
      q_latent_loss = F.mse_loss(quantized, x.detach())
      # gradient flows into `x`
      # moves encoder output towards the codebook
      e_latent_loss = F.mse_loss(quantized.detach(), x)

      # q_latent_loss should move freely toward whatever encoder outputs
      # e_latent_loss is riskier to push that's why i am using commitment_cost to dampen the push and mitigate prematurity
      loss = q_latent_loss + self.commitment_cost * e_latent_loss

      # forward pass let quantized values pass
      # but backward pass must let gradients through as if nothing was quantized
      # gradient of quantized wrt x = gradient of x wrt x = 1 :: gradients pass through untouched
      quantized = x + (quantized-x).detach()
      return quantized, loss

In [ ]:
class ResidualVectorQuantizer(nn.Module):
  def __init__(self, num_codebooks=4, codebook_size=1024, embedding_dim=32, commitment_cost=0.25):
    super().__init__()
    self.codebooks = nn.ModuleList([
        VectorQuantizer(codebook_size, embedding_dim, commitment_cost) for _ in range(num_codebooks)
    ])

  def forward(self, x):
    residual = x
    # accumulator that will collect the sum of every stage's contribution
    out = torch.zeros_like(x)
    # accumulates each stage's VQ loss
    total_loss = 0.0

    for codebook in self.codebooks:
      # VectorQuantizer.forward(residual)
      quantized_diff, loss = codebook(residual)
      out = out + quantized_diff
      residual = residual - quantized_diff
      total_loss = total_loss + loss
    return out, total_loss

In [ ]:
class ResidualDownSampleBlock(nn.Module):
  def __init__(self, in_channels, out_channels, stride, kernel_size=8):
    super().__init__()
    padding_val = (kernel_size - 1) // 2

    self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding_val)
    self.bn1 = nn.BatchNorm1d(out_channels)
    self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, stride=stride, padding=padding_val)
    self.relu = nn.ReLU()

    # skip connection will only work if it'll have same forward moving shape
    if in_channels != out_channels or stride != 1:
      self.residual_proj = nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride)
    else:
      self.residual_proj = nn.Identity() # no op layer

def forward(self, x):
  # skip connection
  residual = self.residual_proj(x)
  out = self.relu(self.bn1(self.conv1(x)))
  out = self.conv2(out)

  # if time length doesnt match
  if out.shape[-1] != residual.shape[-1]:
    min_len = min(out.shape[-1], residual.shape[-1])
    out, residual = out[..., :min_len], residual[..., :min_len]

  return self.relu(out + residual)

In [ ]:
class DownsamplingNetwork(nn.Module):
  # tuple of strides to compress audio to manageable length for Transformer
  def __init__(self, embedding_dim=32, hidden_dim=16, in_channels=1, strides=(2,2,2,2)):
    super().__init__()
    self.mean_pooling = nn.AvgPool1d(kernel_size=2, stride=2)

    self.layers = nn.ModuleList()
    current_in = in_channels

    for i, s in enumerate(strides):
      block_in = current_in if i==0 else hidden_dim
      self.layers.append(ResidualDownSampleBlock(block_in, hidden_dim, stride=s))

    self.final_conv = nn.Conv1d(hidden_dim, embedding_dim, kernel_size=4, padding="same")

  def forward(self, x):
    # extra 2x downsample right at the start (simple averaging, no learned weights)
    x = self.mean_pooling(x)
    for layer in self.layers:
      x = layer(x)
    # we hit 32x compression
    x = self.final_conv(x)
    return x.transpose(1, 2)

```text
most STT systems like Whisper, wav2vec2 skip descretization entirely and go straight from Transformer -> linear -> CTC loss
we are just doing RVQ before CTC loss
```

attention treats input as an unordered set, so im baking in fingerprint per position

In [ ]:
import math

class SinusoidalPositionEncoding(nn.Module):
  def __init__(self, embed_size, max_seq_length=10000): # (max_len, 1)
    super().__init__()
    position = torch.arange(max_seq_length).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, embed_size, 2) * (-math.log(10000.0) / embed_size))
    pe = torch.zeros(max_seq_length, embed_size)
    pe[:, 0::2] = torch.sin(position * div_term) # even dims
    pe[:, 1::2] = torch.cos(position * div_term) # odd dims

    self.register_buffer("positional_embedding", pe) # fixed, not trained

  def forward(self, x):
    return x + self.positional_embedding[:x.size(1), :]

In [ ]:
import torch.nn.functional as F

class FeedForward(nn.Module):
  def __init__(self, embed_size, ff_hidden_mult=4, dropout=0.1):
    self.layer1 = nn.Linear(embed_size, hidden)
    self.layer2 = nn.Linear(hidden, embed_size)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    return self.layer2(self.dropout(F.gelu(self.layer1(x))))

In [ ]:
class SelfAttentionLayer(nn.Module):
  def __init__(self, embed_size, num_heads, dropout=0.1):
    super().__init__()
    self.mha = nn.MultiheadAttention(embed_size, num_heads, dropout=dropout, batch_first=True)
    self.attn_dropout = nn.Dropout(dropout)
    self.attn_norm = nn.LayerNorm(embed_size)
    self.ff = FeedForward(embed_size)
    self.ff_dropout = nn.Dropout(dropout)
    self.ff_norm = nn.LayerNorm(embed_size)

  def forward(self, x):
    attn_out, _ = self.mha(x, x, x, need_weights=False)
    x = self.attn_norm(x + self.attn_dropout(attn_out)))
    ff_out = self.ff(x)
    return self.ff_norm(x + self.ff_dropout(ff_out))

In [ ]:
class TransformerEncoder(nn.Module):
  def __init__(self, embed_size=32, num_layers=6, num_heads=4, max_seq_length=10000):
    super().__init__()
    self.positional_encoding = SinusoidalPositionEncoding(embed_size, max_seq_length)
    self.blocks = nn.ModuleList([SelfAttentionLayer(embed_size, num_heads) for _ in range(num_layers)])
    self.dropout = nn.Dropout(0.1)

  def forward(self, x):
    x = self.dropout(self.positional_encoding(x))
    for block in self.blocks:
      x = block(x)
    return x

In [ ]:
class TranscribeModel(nn.Module):
  def __init__(self, num_codebooks=4, codebook_size=1024, embedding_dim=32, vocab_size=28, strides=(2,2,2,2), num_transformer_layers=6, num_heads=4, max_seq_length=10000):
    super().__init__()
    self.downsampling_network = DownsamplingNetwork(
            embedding_dim=embedding_dim, hidden_dim=embedding_dim // 2,
            in_channels=1, strides=strides
        )
        self.pre_rvq_transformer = TransformerEncoder(
            embed_size=embedding_dim, num_layers=num_transformer_layers,
            num_heads=num_heads, max_seq_length=max_seq_length
        )
        self.rvq = ResidualVectorQuantizer(
            num_codebooks=num_codebooks, codebook_size=codebook_size, embedding_dim=embedding_dim
        )
        self.output_layer = nn.Linear(embedding_dim, vocab_size)

  def forward(self, x):
    if x.dim() == 2:
      x = x.unsqueeze(1) # (B, T) -> (B, 1, T)

    x = self.downsampling_network(x) # (B, T', D)
    x = self.pre_rvq_transformer(x) # (B, T', D)
    x, vq_loss = self.rvq(x) # (B, T', D)
    x = self.output_layer(x) # (B, T', vocab)
    log_probs = F.log_softmax(x, dim=-1)
    return log_probs, vq_loss

In [ ]:
model = TranscribeModel(
    num_codebooks=CFG["num_codebooks"],
    codebook_size=CFG["codebook_size"],
    embedding_dim=CFG["embedding_dim"],
    vocab_size=vocab_size,
    strides=CFG["strides"],
    num_transformer_layers=CFG["num_transformer_layers"],
    num_heads=CFG["num_heads"],
).to(device)

In [ ]:
scaler = torch.amp.GradScaler("cuda")
optimizer = torch.optim.Adam(mode.parameters(), lr=CFG["lr"])
blank_idx = char_to_idx["<blank>"]
ctc_loss_fn = torch.nn.CTCLoss(blank=blank_idx, zero_infinity=True)

num_epochs = CFG["epochs"]
best_loss = float("inf")
steps = 0
os.makedirs("checkpoints", exist_ok=True)

for epoch in range(num_epochs):
  model.train()
  total_loss = 0.0
  for i, (waveforms, tokens, audio_lengths, target_lengths) in enumerate(loader):
    waveforms = waveforms.to(device)
    tokens = tokens.to(device)
    target_lengths = target_lengths.to(device)

    optimizer.zero_grad()

    with torch.cuda.amp.autocast():
      log_probs, vq_loss = model(waveforms)

      input_lengths = torch.full(
          size=(log_probs.shape[0],),
          fill_value = log_probs.shape[1],
          dtype = torch.long,
          device = device
      )

      log_probs_t = log_probs.permute(1, 0 , 2) # (T, B, vocab_size)
      ctc_loss = ctc_loss_fn(log_probs_t, tokens, input_lengths, target_lengths)

      vq_weight = max(
          CFG["vq_final_weight"],
          CFG["vq_initial_weight"] - (CFG["vq_initial_weight"] - CFG["vq_final_weight"]) * (steps / CFG["vq_warmup_steps"])
      )

      loss = ctc_loss + vq_weight * vq_loss

      # if batch somehow produces a broken loss , skip it entirely rather than corrupting with bad gradient step
      if torch.isnan(loss) or torch.isinf(loss):
        continue

      scaler.scale(loss).backward()
      scaler.unscale_(optimizer)
      torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
      scaler.step(optimizer)
      scaler.update()
      steps += 1

      total_loss += loss.item()
      if i % 100 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}] Step [{i}] CTC: {ctc_loss.item():.4f} VQ: {vq_loss.item():.4f}")

  avg_loss = total_loss / len(loader)
  print(f"\nEpoch [{epoch+1}/{num_epochs}] Avg Loss: {avg_loss:.4f}\n")

  torch.save({
      "epoch": epoch,
      "model_state_dict": model.state_dict(),
      "optimizer_state_dict": optimizer.state_dict(),
  }, f"checkpoints/last.pth")

if avg_loss < best_loss:
  best_loss = avg_loss
  torch.save(model.state_dict(), "checkpoints/best_model.pth")
  print("best model saved")

In [ ]:
def greedy_decode(log_probs, blank_idx):
  pred_idx = log_probs.argmax(dim=-1).tolist()
  chars = []
  prev = blank_idx
  for idx in pred_idx:
    if idx != blank_idx and idx != prev:


In [ ]:
import shutil

shutil.rmtree("./data", ignore_errors=True)

for f in os.listdir("."):
  if f.endswith(".tar.bz2"):
    os.remove(f)

print("cleanup done. ", os.listdir("."))